# Pentathlon: Next Product to Buy Models

By Andy Liu


## Setup

Run top to bottom. The notebook loads `data/pentathlon_nptb.parquet` (not included in this repo — see `data/README.md`), fits four purchase-probability models, scores every customer under all seven actions (six messages plus no-message), and compares personalized, best-single-message, random, and no-message policies on expected profit. Uses polars, pyrsm, and scikit-learn.


In [8]:
# update pyrsm to version 2.3.1
!uv --project /opt/base-uv add pyrsm==2.3.1

Resolved 220 packages in 1ms
Audited 203 packages in 1ms


In [9]:
import polars as pl
import pyrsm as rsm

In [10]:
## loading the data - this dataset must NOT be changed
pentathlon_nptb = pl.read_parquet("data/pentathlon_nptb.parquet")
pentathlon_nptb.head()

custid,buyer,total_os,message,age,female,income,education,children,freq_endurance,freq_strength,freq_water,freq_team,freq_backcountry,freq_racquet,training
str,cat,i32,cat,cat,cat,i32,i32,f64,i32,i32,i32,i32,i32,i32,f64
"""U1""","""no""",0,"""team""","""30 to 44""","""no""",55000,19,0.8,0,4,0,4,0,1,1.0
"""U3""","""no""",0,"""backcountry""","""45 to 59""","""no""",35000,22,1.0,0,0,0,1,0,0,0.0
"""U13""","""no""",0,"""endurance""","""45 to 59""","""yes""",45000,33,0.7,0,0,0,0,2,2,1.0
"""U20""","""no""",0,"""water""","""45 to 59""","""yes""",25000,24,0.2,0,0,0,0,0,0,1.0
"""U25""","""no""",0,"""racquet""","""60 and older""","""yes""",65000,32,1.1,1,2,0,0,2,2,0.0


In [11]:
# show dataset description
rsm.md("data/pentathlon_nptb_description.md")

## Pentathon: Next Product To Buy 

The available data is based on the last e-mail sent to each Pentathlon customer. Hence, an observation or row in the data is a "customer-promotional e-mail" pair. The data contains the following basic demographic information available to Pentathlon:

* "age": Customer age(coded in 4 buckets:"less than 30", "30 to 44", "45 to 59", and "60 and older")
* "female": Gender identity coded as Female "yes" or "no"
* "income": Income in Euros, rounded to the nearest EUR5,000
* "education": Percentage of college graduates in the customer's neighborhood, coded from 0-100
* "children": Average number of children in the customer's neighborhood

The data also contains basic historical information about customer purchases, specifically, a department-specific frequency measure.

* "freq_endurance-freq_racquet": Number of purchases in each department in the last year, excluding any purchase in response to the last email.

The key outcome variables are:

* "buyer": Did the customer click on the e-mail and complete a purchase within two days of receiving the e-mail ("yes" or "no")?
* "total_os": Total order size (in Euros) conditional on the customer having purchased (buyer == "yes"). This measures spending across all departments, not just the department that sent the message

> Note: In addition to the six message groups, a seventh group of customers received no promotional e-mails for the duration of the test ("control").

In [ ]:
# Kernel restart recovery: reload data and recreate train/test splits
pentathlon_nptb = pl.read_parquet("data/pentathlon_nptb.parquet")

train = pentathlon_nptb.filter(pl.col("training") == 1)
test  = pentathlon_nptb.filter(pl.col("training") == 0)

freq_cols = [c for c in pentathlon_nptb.columns if c.startswith("freq_")]

pentathlon_nptb.shape, train.shape, test.shape

((600000, 16), (420000, 16), (180000, 16))

---
***Q1+Q2(Logistic Regression)***

In [ ]:
# Fit logistic regression with message main effects + message×freq interactions (columns created explicitly)
freq_cols = [c for c in pentathlon_nptb.columns if c.startswith("freq_")]

messages = (
    pentathlon_nptb.select(pl.col("message").unique())
    .to_series()
    .to_list()
)
messages_no_control = [m for m in messages if m != "control"]

def add_int_cols(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for m in messages_no_control:
        msg_ind = (pl.col("message") == m).cast(pl.Float64)
        for f in freq_cols:
            exprs.append((msg_ind * pl.col(f)).alias(f"int_{m}_{f}"))
    return df.with_columns(exprs)

train_int = add_int_cols(train)

evar_main = ["message", "age", "female", "income", "education", "children"] + freq_cols
int_cols = [c for c in train_int.columns if c.startswith("int_")]

clf_logit_int = rsm.model.logistic(
    data={"train": train_int},
    rvar="buyer",
    lev="yes",
    evar=evar_main + int_cols,
)

clf_logit_int.summary()

Logistic regression (GLM)
Data                 : train
Response variable    : buyer
Level                : yes
Explanatory variables: message, age, female, income, education, children, freq_endurance, freq_strength, freq_water, freq_team, freq_backcountry, freq_racquet, int_endurance_freq_endurance, int_endurance_freq_strength, int_endurance_freq_water, int_endurance_freq_team, int_endurance_freq_backcountry, int_endurance_freq_racquet, int_water_freq_endurance, int_water_freq_strength, int_water_freq_water, int_water_freq_team, int_water_freq_backcountry, int_water_freq_racquet, int_racquet_freq_endurance, int_racquet_freq_strength, int_racquet_freq_water, int_racquet_freq_team, int_racquet_freq_backcountry, int_racquet_freq_racquet, int_strength_freq_endurance, int_strength_freq_strength, int_strength_freq_water, int_strength_freq_team, int_strength_freq_backcountry, int_strength_freq_racquet, int_backcountry_freq_endurance, int_backcountry_freq_strength, int_backcountry_freq_water, 

In [14]:
# Build counterfactual rows on test and recompute interaction columns after changing message
actions = (
    pentathlon_nptb.select(pl.col("message").unique())
    .to_series()
    .to_list()
)

base_test = test.select(
    ["custid", "age", "female", "income", "education", "children"] + freq_cols
)

cf_test = base_test.join(pl.DataFrame({"message": actions}), how="cross").with_row_index("row_id")

cf_test = add_int_cols(cf_test)

In [15]:
# Score P(buy), compute Q1 best message and Q2 shares
pred = clf_logit_int.predict(cf_test)

pred_cols = [c for c in pred.columns if c not in cf_test.columns]
if len(pred_cols) == 1:
    pred_col = pred_cols[0]
elif len(pred.columns) == 1:
    pred_col = pred.columns[0]
else:
    raise ValueError(f"Cannot identify prediction column: {pred.columns}")

pred = pred.rename({pred_col: "p_buy"})
if "row_id" not in pred.columns:
    pred = pred.with_row_index("row_id")

scored = (
    cf_test.select(["row_id", "custid", "message"])
    .join(pred.select(["row_id", "p_buy"]), on="row_id", how="left")
)

best_prob = (
    scored.sort(["custid", "p_buy"], descending=[False, True])
    .group_by("custid")
    .head(1)
    .select(["custid", "message", "p_buy"])
    .rename({"message": "best_msg_prob"})
)

q2_share = (
    best_prob.group_by("best_msg_prob")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum()).alias("share"))
    .sort("share", descending=True)
)

best_prob.head(10), q2_share

(shape: (10, 3)
 ┌──────────┬───────────────┬──────────┐
 │ custid   ┆ best_msg_prob ┆ p_buy    │
 │ ---      ┆ ---           ┆ ---      │
 │ str      ┆ str           ┆ f64      │
 ╞══════════╪═══════════════╪══════════╡
 │ U1397791 ┆ endurance     ┆ 0.014209 │
 │ U1104189 ┆ endurance     ┆ 0.009605 │
 │ U2767077 ┆ endurance     ┆ 0.006075 │
 │ U1321433 ┆ endurance     ┆ 0.007168 │
 │ U1510345 ┆ endurance     ┆ 0.027659 │
 │ U280965  ┆ endurance     ┆ 0.002811 │
 │ U2171439 ┆ endurance     ┆ 0.003516 │
 │ U999166  ┆ endurance     ┆ 0.006567 │
 │ U2623614 ┆ endurance     ┆ 0.006637 │
 │ U2287351 ┆ endurance     ┆ 0.009133 │
 └──────────┴───────────────┴──────────┘,
 shape: (4, 3)
 ┌───────────────┬────────┬──────────┐
 │ best_msg_prob ┆ n      ┆ share    │
 │ ---           ┆ ---    ┆ ---      │
 │ str           ┆ u32    ┆ f64      │
 ╞═══════════════╪════════╪══════════╡
 │ endurance     ┆ 158356 ┆ 0.879756 │
 │ strength      ┆ 18101  ┆ 0.100561 │
 │ water         ┆ 1778   ┆ 0.009878 │


Q1: We fit a logistic regression to estimate P(buyer=yes | X, message). To enable customer-level personalization, we include message-by-history interactions (message × purchase-frequency variables). For each customer, we create counterfactual rows for all messages (including control), score predicted purchase probabilities, and select the message with the highest probability.

Most customers are predicted to respond best to the endurance message (~88%), with a meaningful minority assigned to strength (~10%). Water and team are selected for about 1% each, indicating limited but non-zero heterogeneity in predicted response across messages. Control and the remaining messages are rarely optimal under this model.

---
***Q3(Logistic Regression)***

In [ ]:
# Q3: Max expected profit using logistic p(buy) + statsmodels OLS for log1p(order size)
import numpy as np
import statsmodels.formula.api as smf

margin = 0.4

freq_cols = [c for c in pentathlon_nptb.columns if c.startswith("freq_")]

messages = (
    pentathlon_nptb.select(pl.col("message").unique())
    .to_series()
    .to_list()
)
messages_no_control = [m for m in messages if m != "control"]

def add_int_cols(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for m in messages_no_control:
        msg_ind = (pl.col("message") == m).cast(pl.Float64)
        for f in freq_cols:
            exprs.append((msg_ind * pl.col(f)).alias(f"int_{m}_{f}"))
    return df.with_columns(exprs)

actions = messages

buyers_train = train.filter(pl.col("buyer") == "yes").with_columns(
    pl.col("total_os").cast(pl.Float64).log1p().alias("log_total_os")
)
buyers_train = add_int_cols(buyers_train)

int_cols = [c for c in buyers_train.columns if c.startswith("int_")]
base_cols = ["message", "age", "female", "income", "education", "children"] + freq_cols + int_cols

buyers_pd = buyers_train.select(base_cols + ["log_total_os"]).to_pandas()

formula = "log_total_os ~ C(message) + C(age) + C(female) + income + education + children + " + " + ".join(freq_cols + int_cols)
spend_ols = smf.ols(formula, data=buyers_pd).fit()

base_test = test.select(
    ["custid", "age", "female", "income", "education", "children"] + freq_cols
)
cf_test = base_test.join(pl.DataFrame({"message": actions}), how="cross").with_row_index("row_id")
cf_test = add_int_cols(cf_test)

p_pred = clf_logit_int.predict(cf_test)
p_cols = [c for c in p_pred.columns if c not in cf_test.columns]
p_col = p_cols[0] if len(p_cols) == 1 else p_pred.columns[0]
p_pred = p_pred.rename({p_col: "p_buy"})
if "row_id" not in p_pred.columns:
    p_pred = p_pred.with_row_index("row_id")

cf_pd = cf_test.select(base_cols).to_pandas()
log_m_buy = spend_ols.predict(cf_pd)
m_buy = np.expm1(np.clip(log_m_buy, a_min=0, a_max=None))

m_pred = pl.DataFrame({"row_id": cf_test["row_id"], "m_buy": m_buy})

scored_ep = (
    cf_test.select(["row_id", "custid", "message"])
    .join(p_pred.select(["row_id", "p_buy"]), on="row_id", how="left")
    .join(m_pred, on="row_id", how="left")
    .with_columns((pl.col("p_buy") * margin * pl.col("m_buy")).alias("ep"))
)

best_ep = (
    scored_ep.sort(["custid", "ep"], descending=[False, True])
    .group_by("custid")
    .head(1)
    .select(["custid", "message", "ep"])
    .rename({"message": "best_msg_profit"})
)

best_ep.head(10)

custid,best_msg_profit,ep
str,str,f64
"""U2154061""","""endurance""",0.085131
"""U836393""","""endurance""",0.376737
"""U2533474""","""endurance""",0.112481
"""U1168520""","""endurance""",0.038936
"""U1963338""","""endurance""",0.171308
"""U1251819""","""water""",7.027848
"""U1684880""","""endurance""",0.185082
"""U2184510""","""endurance""",0.013804
"""U2047098""","""endurance""",0.898606


We report the share of customers for whom each message (including control) maximizes expected profit. This summarizes how the profit-maximizing recommendation is distributed across actions.

---
***Q4(Logistic Regression)***

In [17]:
# Q4: Share of customers for whom each message maximizes expected profit
q4_share = (
    best_ep.group_by("best_msg_profit")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum()).alias("share"))
    .sort("share", descending=True)
)

q4_share

best_msg_profit,n,share
str,u32,f64
"""endurance""",152888,0.849378
"""water""",12451,0.069172
"""backcountry""",6448,0.035822
"""team""",5906,0.032811
"""strength""",2307,0.012817


We report, for each message (and the control option), the percentage of customers for whom that choice maximizes expected profit. This summarizes how the profit-maximizing recommendation is distributed across actions and highlights whether the optimal policy is concentrated in a few messages or spread across multiple messages.

---
***Q5(Logistic Regression)***

In [18]:
# Q5: Average expected profit per customer under personalization
q5_avg_ep = best_ep.select(pl.col("ep").mean().alias("avg_ep_per_customer"))
q5_avg_ep

avg_ep_per_customer
f64
0.488103


Under the personalized policy, we assign each customer the message (or control) that maximizes their expected profit and then average that maximum expected profit across customers. Using this approach, the expected profit is 0.4881 Euros per customer on the test set.

---
***Q6(Logistic Regression)***

In [19]:
# Q6: Expected profit per customer if everyone receives the same message (or control)

q6_ep_by_message = (
    scored_ep.group_by("message")
    .agg(pl.col("ep").mean().alias("avg_ep_per_customer"))
    .sort("avg_ep_per_customer", descending=True)
)

q6_ep_by_message

message,avg_ep_per_customer
str,f64
"""endurance""",0.459974
"""water""",0.450305
"""strength""",0.427357
"""backcountry""",0.423026
"""team""",0.39892
"""racquet""",0.390271
"""control""",0.312201


If every customer must receive the same message, the best single-message choice is endurance, with an expected profit of 0.45997 Euros per customer. The next best are water (0.45031) and strength (0.42736), while control/no-message is lowest at 0.31220. Overall, expected profit varies meaningfully across messages, and any message outperforms the no-message option in this model.

---
***Q7(Logistic Regression)***

In [ ]:
# Q7: Expected profit per customer under random assignment to one of 7 actions (incl. control)
q7_random_ep = (
    scored_ep.group_by("message")
    .agg(pl.col("ep").mean().alias("avg_ep"))
    .select((pl.col("avg_ep").mean()).alias("avg_ep_per_customer_random"))
)

q7_random_ep

avg_ep_per_customer_random
f64
0.408865


Under random assignment (each customer equally likely to receive any of the 6 messages or the control option), the expected profit is 0.4089 Euros per customer. This is lower than sending the best single message to everyone (Q6) and lower than personalization (Q5), because random assignment frequently allocates customers to lower-performing messages and the no-message condition.

---
***Q8(Logistic Regression)***

In [21]:
# Q8: Scale expected profit to 5,000,000 customers and compare scenarios

N = 5_000_000

ep_personal = 0.488103   # from Q5
ep_best_one = 0.459974   # best single message from Q6 (endurance)
ep_random   = 0.408865   # from Q7
ep_control  = 0.312201   # control from Q6 table

summary = pl.DataFrame({
    "scenario": ["personalized", "best_single_message", "random_assignment", "control_no_message"],
    "ep_per_customer": [ep_personal, ep_best_one, ep_random, ep_control],
}).with_columns(
    (pl.col("ep_per_customer") * N).alias("total_ep_5m")
)

summary

scenario,ep_per_customer,total_ep_5m
str,f64,f64
"""personalized""",0.488103,2.440515e6
"""best_single_message""",0.459974,2.29987e6
"""random_assignment""",0.408865,2.044325e6
"""control_no_message""",0.312201,1.561005e6


In [22]:
# Q8: Improvements of personalization vs (a) best single, (b) random, (c) control

N = 5_000_000
ep_personal = 0.488103
comparators = {
    "best_single_message": 0.459974,
    "random_assignment": 0.408865,
    "control_no_message": 0.312201,
}

impr = []
for name, base in comparators.items():
    euro_gain = (ep_personal - base) * N
    pct_gain = (ep_personal - base) / base
    impr.append((name, base, euro_gain, pct_gain))

improvement = pl.DataFrame(impr, schema=["baseline", "baseline_ep_per_customer", "euro_gain_5m", "pct_gain"])
improvement

/tmp/ipykernel_3020/4163600484.py:17: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.


baseline,baseline_ep_per_customer,euro_gain_5m,pct_gain
str,f64,f64,f64
"""best_single_message""",0.459974,140645.0,0.061153
"""random_assignment""",0.408865,396190.0,0.1938
"""control_no_message""",0.312201,879510.0,0.563425


Using the test-set expected profit estimates, we scale to a typical blast of 5,000,000 customers by multiplying expected profit per customer by 5,000,000. The resulting total expected profit is:

Personalized: 0.488103 × 5,000,000 = €2,440,515

Best single message (endurance): 0.459974 × 5,000,000 = €2,299,870

Random assignment (status quo): 0.408865 × 5,000,000 = €2,044,325

Control (no message): 0.312201 × 5,000,000 = €1,561,005

Improvements from personalization:

vs best single message: +€140,645 (+6.12%)

vs random assignment: +€396,190 (+19.38%)

vs control/no message: +€879,510 (+56.34%)

Overall, personalization delivers the highest expected profit and provides meaningful gains over the status quo (random assignment) and the no-message baseline, while still improving over the best “one-size-fits-all” message.

---
***Q1(NN)***

In [ ]:
# NN Q1: Best message by predicted purchase probability (tuned hidden_layer_sizes + alpha)
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import GridSearchCV

actions = (
    pentathlon_nptb.select(pl.col("message").unique())
    .to_series()
    .to_list()
)

cat_cols = ["message", "age", "female"]
num_cols = ["income", "education", "children"] + [c for c in pentathlon_nptb.columns if c.startswith("freq_")]
feature_cols = cat_cols + num_cols

train_pd = train.select(feature_cols + ["buyer"]).to_pandas()
X_train = train_pd[feature_cols]
y_train = (train_pd["buyer"] == "yes").astype(int)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ]
)

mlp = MLPClassifier(
    random_state=123,
    max_iter=200,
    early_stopping=True,
)

pipe = Pipeline([("prep", preprocess), ("mlp", mlp)])

param_grid = {
    "mlp__hidden_layer_sizes": [(32, 16), (32, 32), (64, 32)],
    "mlp__alpha": [1e-5, 1e-4, 1e-3],
}

gs = GridSearchCV(pipe, param_grid=param_grid, scoring="roc_auc", cv=3, n_jobs=-1)
gs.fit(X_train, y_train)

best_nn = gs.best_estimator_

base_test = test.select(
    ["custid", "age", "female", "income", "education", "children"] + [c for c in pentathlon_nptb.columns if c.startswith("freq_")]
)

cf_test = base_test.join(pl.DataFrame({"message": actions}), how="cross")

cf_pd = cf_test.select(feature_cols).to_pandas()
p_buy = best_nn.predict_proba(cf_pd)[:, 1]

scored_nn = cf_test.select(["custid", "message"]).with_columns(
    pl.Series("p_buy", p_buy)
)

best_prob_nn = (
    scored_nn.sort(["custid", "p_buy"], descending=[False, True])
    .group_by("custid")
    .head(1)
    .select(["custid", "message", "p_buy"])
    .rename({"message": "best_msg_prob_nn"})
)

best_prob_nn.head(10)

custid,best_msg_prob_nn,p_buy
str,str,f64
"""U332756""","""endurance""",0.003164
"""U1230217""","""endurance""",0.008724
"""U2733669""","""endurance""",0.001642
"""U416976""","""strength""",0.008962
"""U3360850""","""endurance""",0.003008
"""U542091""","""strength""",0.009511
"""U1689416""","""endurance""",0.004378
"""U1748248""","""strength""",0.137836
"""U1808646""","""endurance""",0.053518


We fit a neural network (MLP) to predict P(buyer=yes | X, message). Categorical variables (message, age, female) are one-hot encoded and numeric variables are standardized. The model is tuned via cross-validation using two hyperparameters—hidden_layer_sizes and alpha. For each customer, we score all 7 messages (including control) and select the message with the highest predicted purchase probability.

---
***Q2(NN)***

In [24]:
# NN Q2: Share of customers for whom each message maximizes predicted purchase probability
q2_share_nn = (
    best_prob_nn.group_by("best_msg_prob_nn")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum()).alias("share"))
    .sort("share", descending=True)
)

q2_share_nn

best_msg_prob_nn,n,share
str,u32,f64
"""endurance""",113285,0.629361
"""strength""",43349,0.240828
"""team""",16108,0.089489
"""water""",3388,0.018822
"""racquet""",2875,0.015972
"""backcountry""",995,0.005528


Under the tuned neural network model, the recommended message distribution is more diverse than the baseline logistic model. Endurance is still most often optimal (62.94%), but a substantial share of customers are assigned to strength (24.08%) and team (8.95%), with smaller segments assigned to water (1.88%), racquet (1.60%), and backcountry (0.55%). This indicates stronger customer-level heterogeneity in predicted purchase response across messages.

---
***Q3(NN)***

In [25]:
# NN Q3: Best message by expected profit (uses NN p_buy + same spend OLS model)
margin = 0.4

actions = (
    pentathlon_nptb.select(pl.col("message").unique())
    .to_series()
    .to_list()
)

freq_cols = [c for c in pentathlon_nptb.columns if c.startswith("freq_")]
messages_no_control = [m for m in actions if m != "control"]

def add_int_cols(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for m in messages_no_control:
        msg_ind = (pl.col("message") == m).cast(pl.Float64)
        for f in freq_cols:
            exprs.append((msg_ind * pl.col(f)).alias(f"int_{m}_{f}"))
    return df.with_columns(exprs)

# --- Spend model (buyers only): OLS on log1p(total_os)
buyers_train = train.filter(pl.col("buyer") == "yes").with_columns(
    pl.col("total_os").cast(pl.Float64).log1p().alias("log_total_os")
)
buyers_train = add_int_cols(buyers_train)

int_cols = [c for c in buyers_train.columns if c.startswith("int_")]
spend_cols = ["message", "age", "female", "income", "education", "children"] + freq_cols + int_cols

buyers_pd = buyers_train.select(spend_cols + ["log_total_os"]).to_pandas()
formula = "log_total_os ~ C(message) + C(age) + C(female) + income + education + children + " + " + ".join(freq_cols + int_cols)
spend_ols = smf.ols(formula, data=buyers_pd).fit()

# --- Counterfactual rows on test: customers × messages
cat_cols = ["message", "age", "female"]
num_cols = ["income", "education", "children"] + freq_cols
feature_cols = cat_cols + num_cols

base_test = test.select(["custid", "age", "female", "income", "education", "children"] + freq_cols)
cf_test = base_test.join(pl.DataFrame({"message": actions}), how="cross")

# --- NN p_buy (requires best_nn from NN Q1)
cf_pd = cf_test.select(feature_cols).to_pandas()
p_buy = best_nn.predict_proba(cf_pd)[:, 1]

# --- Spend prediction m_buy
cf_test_int = add_int_cols(cf_test)
cf_pd_spend = cf_test_int.select(spend_cols).to_pandas()
log_m_buy = spend_ols.predict(cf_pd_spend)
m_buy = np.expm1(np.clip(log_m_buy, a_min=0, a_max=None))

# --- Expected profit and best message
scored_ep_nn = cf_test.select(["custid", "message"]).with_columns(
    pl.Series("p_buy", p_buy),
    pl.Series("m_buy", m_buy),
).with_columns(
    (pl.col("p_buy") * margin * pl.col("m_buy")).alias("ep")
)

best_ep_nn = (
    scored_ep_nn.sort(["custid", "ep"], descending=[False, True])
    .group_by("custid")
    .head(1)
    .select(["custid", "message", "ep"])
    .rename({"message": "best_msg_profit_nn"})
)

best_ep_nn.head(10)

custid,best_msg_profit_nn,ep
str,str,f64
"""U994853""","""endurance""",0.022017
"""U1072229""","""endurance""",0.017896
"""U3302180""","""team""",0.023162
"""U2625486""","""water""",0.781824
"""U3261739""","""endurance""",0.103769
"""U1212447""","""team""",0.210358
"""U3295256""","""endurance""",0.022994
"""U320917""","""endurance""",0.665608
"""U2794492""","""endurance""",0.225593


We compute expected profit by combining (1) the neural network’s predicted purchase probability P(buyer=yes | X, message) with (2) a conditional order-size model estimated on buyers only. We fit an OLS regression for log1p(total_os) using customer features and message indicators (including message-by-frequency interaction terms), back-transform predictions to Euros, and calculate expected profit as:

EP = P(buy) × 0.4 × E(total_os | buy) (COGS = 60% ⇒ margin = 40%).

For each customer, we score all messages (including the control/no-message option) and select the message that maximizes expected profit.

---
***Q4(NN)***

In [26]:
# NN Q4: Share of customers for whom each message maximizes expected profit

q4_share_nn = (
    best_ep_nn.group_by("best_msg_profit_nn")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum()).alias("share"))
    .sort("share", descending=True)
)

q4_share_nn

best_msg_profit_nn,n,share
str,u32,f64
"""endurance""",110465,0.613694
"""team""",24951,0.138617
"""water""",14757,0.081983
"""strength""",12042,0.0669
"""backcountry""",10429,0.057939
"""racquet""",7356,0.040867


Under the neural-network profit-maximizing policy, recommendations are more dispersed across messages than in the logistic baseline. Endurance is still the most frequently optimal message (61.37%), but sizable segments of customers are assigned to team (13.86%) and water (8.20%), with additional customers assigned to strength (6.69%), backcountry (5.79%), and racquet (4.09%). This distribution indicates meaningful heterogeneity in which message maximizes expected profit across customers.

---
***Q5(NN)***

In [27]:
# NN Q5: Average expected profit per customer under personalization
q5_avg_ep_nn = best_ep_nn.select(pl.col("ep").mean().alias("avg_ep_per_customer"))
q5_avg_ep_nn

avg_ep_per_customer
f64
0.516939


Under the personalized neural-network policy, we assign each customer the message (or control) that maximizes their expected profit and then average this maximum expected profit across customers. The resulting expected profit is 0.5169 Euros per customer on the test set.

---
***Q6(NN)***

In [28]:
# NN Q6: Expected profit per customer if everyone receives the same message (or control)
q6_ep_by_message_nn = (
    scored_ep_nn.group_by("message")
    .agg(pl.col("ep").mean().alias("avg_ep_per_customer"))
    .sort("avg_ep_per_customer", descending=True)
)

q6_ep_by_message_nn

message,avg_ep_per_customer
str,f64
"""endurance""",0.482611
"""water""",0.465369
"""strength""",0.445003
"""backcountry""",0.438657
"""team""",0.430963
"""racquet""",0.407242
"""control""",0.316215


If every customer must receive the same message, the best single-message choice is endurance, with an expected profit of 0.4826 Euros per customer. The next best are water (0.4654) and strength (0.4450), while control/no-message is lowest at 0.3162. Overall, there is meaningful variation across messages, and the best one-size-fits-all option still underperforms the personalized NN policy (Q5).

---
***Q7(NN)***

In [ ]:
# NN Q7: Expected profit per customer under random assignment to one of 7 actions (incl. control)
q7_random_ep_nn = (
    scored_ep_nn.group_by("message")
    .agg(pl.col("ep").mean().alias("avg_ep"))
    .select(pl.col("avg_ep").mean().alias("avg_ep_per_customer_random"))
)

q7_random_ep_nn

avg_ep_per_customer_random
f64
0.42658


Under random assignment (each customer equally likely to receive any of the 6 messages or the control option), the expected profit is 0.4266 Euros per customer. This is lower than the best single-message policy (Q6) and substantially lower than the personalized NN policy (Q5), since random assignment frequently allocates customers to lower-performing messages and the no-message condition.

---
***Q8(NN)***

In [ ]:
# NN Q8: Scale to 5,000,000 customers and compute improvements vs (a) best single, (b) random, (c) control
N = 5_000_000

ep_personal = 0.516939
ep_best_one = 0.482611   # best single message from Q6 (endurance)
ep_random   = 0.42658
ep_control  = 0.316215

summary_nn = pl.DataFrame({
    "scenario": ["personalized", "best_single_message", "random_assignment", "control_no_message"],
    "ep_per_customer": [ep_personal, ep_best_one, ep_random, ep_control],
}).with_columns(
    (pl.col("ep_per_customer") * N).alias("total_ep_5m")
)

improvement_nn = pl.DataFrame({
    "baseline": ["best_single_message", "random_assignment", "control_no_message"],
    "baseline_ep_per_customer": [ep_best_one, ep_random, ep_control],
}).with_columns(
    ((ep_personal - pl.col("baseline_ep_per_customer")) * N).alias("euro_gain_5m"),
    ((ep_personal - pl.col("baseline_ep_per_customer")) / pl.col("baseline_ep_per_customer")).alias("pct_gain")
)

summary_nn, improvement_nn

(shape: (4, 3)
 ┌─────────────────────┬─────────────────┬─────────────┐
 │ scenario            ┆ ep_per_customer ┆ total_ep_5m │
 │ ---                 ┆ ---             ┆ ---         │
 │ str                 ┆ f64             ┆ f64         │
 ╞═════════════════════╪═════════════════╪═════════════╡
 │ personalized        ┆ 0.516939        ┆ 2.584695e6  │
 │ best_single_message ┆ 0.482611        ┆ 2.413055e6  │
 │ random_assignment   ┆ 0.42658         ┆ 2.1329e6    │
 │ control_no_message  ┆ 0.316215        ┆ 1.5811e6    │
 └─────────────────────┴─────────────────┴─────────────┘,
 shape: (3, 4)
 ┌─────────────────────┬──────────────────────────┬──────────────┬──────────┐
 │ baseline            ┆ baseline_ep_per_customer ┆ euro_gain_5m ┆ pct_gain │
 │ ---                 ┆ ---                      ┆ ---          ┆ ---      │
 │ str                 ┆ f64                      ┆ f64          ┆ f64      │
 ╞═════════════════════╪══════════════════════════╪══════════════╪══════════╡
 │ best_s

Using the test-set expected profit estimates, we scale to a typical blast of 5,000,000 customers by multiplying expected profit per customer by 5,000,000. The resulting total expected profit is:

Personalized: 0.516939 × 5,000,000 = €2,584,695

Best single message (endurance): 0.482611 × 5,000,000 = €2,413,055

Random assignment (status quo): 0.42658 × 5,000,000 = €2,132,900

Control (no message): 0.316215 × 5,000,000 = €1,581,075

Improvements from personalization:

vs best single message: +€171,640 (+7.11%)

vs random assignment: +€451,795 (+21.18%)

vs control/no message: +€1,003,620 (+63.48%)

Overall, the personalized neural-network policy delivers the highest expected profit and provides substantial gains over the status quo (random assignment) and the no-message baseline, while still improving over the best one-size-fits-all message.

---
***Q1(RF)***

In [ ]:
# RF Q1: GridSearchCV (3-fold) on stratified subsample, then refit on full train and score best message
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit

actions = (
    pentathlon_nptb.select(pl.col("message").unique())
    .to_series()
    .to_list()
)

cat_cols = ["message", "age", "female"]
freq_cols = [c for c in pentathlon_nptb.columns if c.startswith("freq_")]
num_cols = ["income", "education", "children"] + freq_cols
feature_cols = cat_cols + num_cols

train_pd = train.select(feature_cols + ["buyer"]).to_pandas()
X_full = train_pd[feature_cols]
y_full = (train_pd["buyer"] == "yes").astype(int)

# Stratified subsample for tuning
sss = StratifiedShuffleSplit(n_splits=1, train_size=60_000, random_state=123)
idx_sub, _ = next(sss.split(X_full, y_full))
X_sub = X_full.iloc[idx_sub].copy()
y_sub = y_full.iloc[idx_sub].copy()

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

rf = RandomForestClassifier(random_state=123, n_jobs=1)

pipe = Pipeline([("prep", preprocess), ("rf", rf)])

# Tune >=2 hyperparameters (here: n_estimators and max_depth; max_features optional but included)
param_grid = {
    "rf__n_estimators": [200, 400],
    "rf__max_depth": [None, 15],
    "rf__max_features": ["sqrt", 0.5],
}

gs = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=1,
)

gs.fit(X_sub, y_sub)

best_params_rf = gs.best_params_
best_cv_auc_rf = gs.best_score_

# Refit on full training set using best params
best_rf = Pipeline([
    ("prep", preprocess),
    ("rf", RandomForestClassifier(
        random_state=123,
        n_jobs=1,
        n_estimators=best_params_rf["rf__n_estimators"],
        max_depth=best_params_rf["rf__max_depth"],
        max_features=best_params_rf["rf__max_features"],
    ))
])

best_rf.fit(X_full, y_full)

# Q1 scoring on test: customers x messages
base_test = test.select(["custid", "age", "female", "income", "education", "children"] + freq_cols)
cf_test = base_test.join(pl.DataFrame({"message": actions}), how="cross")

cf_pd = cf_test.select(feature_cols).to_pandas()
p_buy = best_rf.predict_proba(cf_pd)[:, 1]

scored_rf = cf_test.select(["custid", "message"]).with_columns(pl.Series("p_buy", p_buy))

best_prob_rf = (
    scored_rf.sort(["custid", "p_buy"], descending=[False, True])
    .group_by("custid")
    .head(1)
    .select(["custid", "message", "p_buy"])
    .rename({"message": "best_msg_prob_rf"})
)

best_params_rf, best_cv_auc_rf, best_prob_rf.head(10)

({'rf__max_depth': 15, 'rf__max_features': 'sqrt', 'rf__n_estimators': 400},
 np.float64(0.8767710467896176),
 shape: (10, 3)
 ┌──────────┬──────────────────┬──────────┐
 │ custid   ┆ best_msg_prob_rf ┆ p_buy    │
 │ ---      ┆ ---              ┆ ---      │
 │ str      ┆ str              ┆ f64      │
 ╞══════════╪══════════════════╪══════════╡
 │ U3294049 ┆ endurance        ┆ 0.002069 │
 │ U696488  ┆ strength         ┆ 0.039022 │
 │ U2727551 ┆ endurance        ┆ 0.077053 │
 │ U2998669 ┆ control          ┆ 0.005601 │
 │ U2391228 ┆ control          ┆ 0.00753  │
 │ U2450516 ┆ endurance        ┆ 0.026028 │
 │ U69474   ┆ racquet          ┆ 0.004389 │
 │ U2929695 ┆ strength         ┆ 0.05764  │
 │ U1006091 ┆ control          ┆ 0.007683 │
 │ U1865208 ┆ strength         ┆ 0.032964 │
 └──────────┴──────────────────┴──────────┘)

We trained a random forest model to predict P(buyer=yes | X, message) using one-hot encoding for categorical variables (message, age, female). The model was tuned with GridSearchCV using 3-fold cross-validation, tuning at least two hyperparameters (e.g., n_estimators, max_depth, and max_features) and selecting the configuration with the highest cross-validated AUC. For each customer, we scored all seven actions (six messages plus control) and selected the message with the highest predicted purchase probability.

---
***Q2(RF)***

In [ ]:
# RF Q2: Share of customers for whom each message maximizes predicted purchase probability
q2_share_rf = (
    best_prob_rf.group_by("best_msg_prob_rf")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum()).alias("share"))
    .sort("share", descending=True)
)

q2_share_rf

best_msg_prob_rf,n,share
str,u32,f64
"""endurance""",77342,0.429678
"""strength""",32319,0.17955
"""water""",15702,0.087233
"""backcountry""",15095,0.083861
"""team""",14421,0.080117
"""racquet""",14164,0.078689
"""control""",10957,0.060872


Under the tuned random forest model, the recommended message distribution is more diverse than the logistic baseline. Endurance is most often optimal (42.97%), followed by strength (17.96%). The remaining messages are each selected for meaningful segments—water (8.72%), backcountry (8.39%), team (8.01%), and racquet (7.87%)—and the control/no-message option is selected for 6.09% of customers. This indicates substantial heterogeneity in predicted purchase response across messages.

---
***Q3(RF)***

In [ ]:
# RF Q3: Best message by expected profit (uses RF p_buy + same spend OLS model)
import statsmodels.formula.api as smf

margin = 0.4

actions = (
    pentathlon_nptb.select(pl.col("message").unique())
    .to_series()
    .to_list()
)

freq_cols = [c for c in pentathlon_nptb.columns if c.startswith("freq_")]
messages_no_control = [m for m in actions if m != "control"]

def add_int_cols(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for m in messages_no_control:
        msg_ind = (pl.col("message") == m).cast(pl.Float64)
        for f in freq_cols:
            exprs.append((msg_ind * pl.col(f)).alias(f"int_{m}_{f}"))
    return df.with_columns(exprs)

# Spend model (buyers only): OLS on log1p(total_os)
buyers_train = train.filter(pl.col("buyer") == "yes").with_columns(
    pl.col("total_os").cast(pl.Float64).log1p().alias("log_total_os")
)
buyers_train = add_int_cols(buyers_train)

int_cols = [c for c in buyers_train.columns if c.startswith("int_")]
spend_cols = ["message", "age", "female", "income", "education", "children"] + freq_cols + int_cols

buyers_pd = buyers_train.select(spend_cols + ["log_total_os"]).to_pandas()
formula = "log_total_os ~ C(message) + C(age) + C(female) + income + education + children + " + " + ".join(freq_cols + int_cols)
spend_ols = smf.ols(formula, data=buyers_pd).fit()

# Counterfactual rows on test: customers × messages
cat_cols = ["message", "age", "female"]
num_cols = ["income", "education", "children"] + freq_cols
feature_cols = cat_cols + num_cols

base_test = test.select(["custid", "age", "female", "income", "education", "children"] + freq_cols)
cf_test = base_test.join(pl.DataFrame({"message": actions}), how="cross")

# RF p_buy (requires best_rf from RF Q1)
cf_pd = cf_test.select(feature_cols).to_pandas()
p_buy = best_rf.predict_proba(cf_pd)[:, 1]

# Spend prediction m_buy
cf_test_int = add_int_cols(cf_test)
cf_pd_spend = cf_test_int.select(spend_cols).to_pandas()
log_m_buy = spend_ols.predict(cf_pd_spend)
m_buy = np.expm1(np.clip(log_m_buy, a_min=0, a_max=None))

# Expected profit and best message
scored_ep_rf = cf_test.select(["custid", "message"]).with_columns(
    pl.Series("p_buy", p_buy),
    pl.Series("m_buy", m_buy),
).with_columns(
    (pl.col("p_buy") * margin * pl.col("m_buy")).alias("ep")
)

best_ep_rf = (
    scored_ep_rf.sort(["custid", "ep"], descending=[False, True])
    .group_by("custid")
    .head(1)
    .select(["custid", "message", "ep"])
    .rename({"message": "best_msg_profit_rf"})
)

best_ep_rf.head(10)

custid,best_msg_profit_rf,ep
str,str,f64
"""U3333997""","""water""",3.121819
"""U90164""","""water""",0.03447
"""U1790757""","""team""",0.047349
"""U303835""","""endurance""",0.092418
"""U3388766""","""water""",0.019291
"""U1519940""","""endurance""",0.043677
"""U2775529""","""racquet""",0.031441
"""U2098915""","""strength""",3.825971
"""U3288914""","""control""",0.021491


We compute expected profit by combining (1) the random forest’s predicted purchase probability P(buyer=yes | X, message) with (2) a conditional order-size model estimated on buyers only. We fit an OLS regression for log1p(total_os) using customer features and message indicators (including message-by-frequency interaction terms), back-transform predictions to Euros, and calculate expected profit as EP = P(buy) × 0.4 × E(total_os | buy) (COGS = 60% ⇒ margin = 40%). For each customer, we score all messages (including control/no-message) and select the message that maximizes expected profit.

---
***Q4(RF)***

In [ ]:
# RF Q4: Share of customers for whom each message maximizes expected profit
q4_share_rf = (
    best_ep_rf.group_by("best_msg_profit_rf")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum()).alias("share"))
    .sort("share", descending=True)
)

q4_share_rf

best_msg_profit_rf,n,share
str,u32,f64
"""endurance""",60636,0.336867
"""water""",36839,0.204661
"""backcountry""",34692,0.192733
"""racquet""",16467,0.091483
"""strength""",13706,0.076144
"""team""",13473,0.07485
"""control""",4187,0.023261


Under the random-forest profit-maximizing policy, recommendations are more dispersed across messages than in the logistic baseline. Endurance is most frequently optimal (33.69%), but large segments of customers are assigned to water (20.47%) and backcountry (19.27%), with additional customers assigned to racquet (9.15%), strength (7.61%), and team (7.49%). The control/no-message option is optimal for 2.33% of customers. This distribution indicates substantial heterogeneity in which message maximizes expected profit.

---
***Q5(RF)***

In [35]:
# RF Q5: Average expected profit per customer under personalization
q5_avg_ep_rf = best_ep_rf.select(pl.col("ep").mean().alias("avg_ep_per_customer"))
q5_avg_ep_rf

avg_ep_per_customer
f64
0.496616


Under the personalized random-forest policy, we assign each customer the message (or control) that maximizes their expected profit and then average this maximum expected profit across customers. The resulting expected profit is 0.4966 Euros per customer on the test set.

---
***Q6(RF)***

In [ ]:
# RF Q6: Expected profit per customer if everyone receives the same message (or control)
q6_ep_by_message_rf = (
    scored_ep_rf.group_by("message")
    .agg(pl.col("ep").mean().alias("avg_ep_per_customer"))
    .sort("avg_ep_per_customer", descending=True)
)

q6_ep_by_message_rf

message,avg_ep_per_customer
str,f64
"""water""",0.444125
"""backcountry""",0.429149
"""endurance""",0.427441
"""strength""",0.402129
"""team""",0.398155
"""racquet""",0.394954
"""control""",0.328726


If every customer must receive the same message, the best single-message choice is water, with an expected profit of 0.4441 Euros per customer. The next best are backcountry (0.4291) and endurance (0.4274), while control/no-message is lowest at 0.3287. Overall, expected profit varies across messages, and the best one-size-fits-all message still underperforms the personalized RF policy (Q5).

---
***Q7(RF)***

In [37]:
# RF Q7: Expected profit per customer under random assignment to one of 7 actions (incl. control)

q7_random_ep_rf = (
    scored_ep_rf.group_by("message")
    .agg(pl.col("ep").mean().alias("avg_ep"))
    .select(pl.col("avg_ep").mean().alias("avg_ep_per_customer_random"))
)

q7_random_ep_rf

avg_ep_per_customer_random
f64
0.403526


Under random assignment (each customer equally likely to receive any of the 6 messages or the control option), the expected profit is 0.4035 Euros per customer. This is lower than the best single-message policy (Q6) and lower than the personalized RF policy (Q5), since random assignment frequently allocates customers to lower-performing messages and the no-message condition.

---
***Q8(RF)***

In [38]:
# RF Q8: Scale to 5,000,000 customers and compute improvements vs (a) best single, (b) random, (c) control

N = 5_000_000

ep_personal = 0.496616
ep_best_one = 0.444125   # best single message from Q6 (water)
ep_random   = 0.403526
ep_control  = 0.328726

summary_rf = pl.DataFrame({
    "scenario": ["personalized", "best_single_message", "random_assignment", "control_no_message"],
    "ep_per_customer": [ep_personal, ep_best_one, ep_random, ep_control],
}).with_columns(
    (pl.col("ep_per_customer") * N).alias("total_ep_5m")
)

improvement_rf = pl.DataFrame({
    "baseline": ["best_single_message", "random_assignment", "control_no_message"],
    "baseline_ep_per_customer": [ep_best_one, ep_random, ep_control],
}).with_columns(
    ((ep_personal - pl.col("baseline_ep_per_customer")) * N).alias("euro_gain_5m"),
    ((ep_personal - pl.col("baseline_ep_per_customer")) / pl.col("baseline_ep_per_customer")).alias("pct_gain")
)

summary_rf, improvement_rf

(shape: (4, 3)
 ┌─────────────────────┬─────────────────┬─────────────┐
 │ scenario            ┆ ep_per_customer ┆ total_ep_5m │
 │ ---                 ┆ ---             ┆ ---         │
 │ str                 ┆ f64             ┆ f64         │
 ╞═════════════════════╪═════════════════╪═════════════╡
 │ personalized        ┆ 0.496616        ┆ 2.48308e6   │
 │ best_single_message ┆ 0.444125        ┆ 2.220625e6  │
 │ random_assignment   ┆ 0.403526        ┆ 2.01763e6   │
 │ control_no_message  ┆ 0.328726        ┆ 1.64363e6   │
 └─────────────────────┴─────────────────┴─────────────┘,
 shape: (3, 4)
 ┌─────────────────────┬──────────────────────────┬──────────────┬──────────┐
 │ baseline            ┆ baseline_ep_per_customer ┆ euro_gain_5m ┆ pct_gain │
 │ ---                 ┆ ---                      ┆ ---          ┆ ---      │
 │ str                 ┆ f64                      ┆ f64          ┆ f64      │
 ╞═════════════════════╪══════════════════════════╪══════════════╪══════════╡
 │ best_s

Using the test-set expected profit estimates, we scale to a typical blast of 5,000,000 customers by multiplying expected profit per customer by 5,000,000. The resulting total expected profit is:

Personalized: 0.496616 × 5,000,000 = €2,483,080

Best single message (water): 0.444125 × 5,000,000 = €2,220,625

Random assignment (status quo): 0.403526 × 5,000,000 = €2,017,630

Control (no message): 0.328726 × 5,000,000 = €1,643,630

Improvements from personalization:

vs best single message: +€262,455 (+11.82%)

vs random assignment: +€465,450 (+23.07%)

vs control/no message: +€839,450 (+51.07%)

Overall, the personalized random-forest policy yields the highest expected profit and provides meaningful gains over the status quo (random assignment) and the no-message baseline, while also improving over the best one-size-fits-all message.

---
***Q1(XGBOOST)***

In [ ]:
# XGB (pyrsm) Q1: Best message by predicted purchase probability (CV-tuned on >=2 hyperparameters)
from sklearn.metrics import roc_auc_score

# --- Setup
actions = (
    pentathlon_nptb.select(pl.col("message").unique())
    .to_series()
    .to_list()
)

freq_cols = [c for c in pentathlon_nptb.columns if c.startswith("freq_")]
evar = ["message", "age", "female", "income", "education", "children"] + freq_cols

# --- Find the pyrsm XGBoost model constructor (name differs by version)
xgb_ctor = None
for name in ["xgboost", "xgb", "xgb_classifier", "xgbc"]:
    if hasattr(rsm.model, name):
        xgb_ctor = getattr(rsm.model, name)
        xgb_name = name
        break
if xgb_ctor is None:
    raise AttributeError("Could not find an XGBoost model in rsm.model (tried xgboost/xgb/xgb_classifier/xgbc).")

# --- Simple K-fold CV (no pandas; AUC computed with sklearn.metrics)
K = 3
train_cv = train.with_row_index("row_id").with_columns((pl.col("row_id") % K).alias("fold"))

# Tune >=2 hyperparameters (here: max_depth and n_estimators; plus learning_rate)
param_grid = [
    {"max_depth": 3, "n_estimators": 300, "learning_rate": 0.1},
    {"max_depth": 3, "n_estimators": 600, "learning_rate": 0.05},
    {"max_depth": 6, "n_estimators": 300, "learning_rate": 0.1},
    {"max_depth": 6, "n_estimators": 600, "learning_rate": 0.05},
]

def fit_xgb(train_df: pl.DataFrame, params: dict):
    # pyrsm model APIs vary; we try common patterns
    try:
        return xgb_ctor(data={"train": train_df}, rvar="buyer", lev="yes", evar=evar, **params)
    except TypeError:
        # some versions use 'params=' dict
        return xgb_ctor(data={"train": train_df}, rvar="buyer", lev="yes", evar=evar, params=params)

def predict_proba(model, df: pl.DataFrame) -> pl.Series:
    pred = model.predict(df)
    # find prediction column (the one not in df)
    pred_cols = [c for c in pred.columns if c not in df.columns]
    if len(pred_cols) == 1:
        pc = pred_cols[0]
        return pred[pc]
    # otherwise assume single column output
    if len(pred.columns) == 1:
        return pred[pred.columns[0]]
    raise ValueError(f"Cannot identify prediction column from: {pred.columns}")

best_params = None
best_auc = -1.0

for params in param_grid:
    aucs = []
    for k in range(K):
        tr = train_cv.filter(pl.col("fold") != k).drop(["row_id", "fold"])
        va = train_cv.filter(pl.col("fold") == k).drop(["row_id", "fold"])
        m = fit_xgb(tr, params)
        p = predict_proba(m, va).to_numpy()
        y = (va["buyer"] == "yes").to_numpy().astype(int)
        aucs.append(roc_auc_score(y, p))
    mean_auc = float(np.mean(aucs))
    if mean_auc > best_auc:
        best_auc = mean_auc
        best_params = params

best_params, best_auc

({'max_depth': 3, 'n_estimators': 600, 'learning_rate': 0.05},
 0.8863863446658794)

In [ ]:
# XGB Q1 scoring on TEST: customers × messages, pick best message per customer
# Fit on full training data with best_params
xgb_best = None
try:
    xgb_best = xgb_ctor(data={"train": train}, rvar="buyer", lev="yes", evar=evar, **best_params)
except TypeError:
    xgb_best = xgb_ctor(data={"train": train}, rvar="buyer", lev="yes", evar=evar, params=best_params)

base_test = test.select(
    ["custid", "age", "female", "income", "education", "children"] + freq_cols
)
cf_test = base_test.join(pl.DataFrame({"message": actions}), how="cross")

p_buy = predict_proba(xgb_best, cf_test).rename("p_buy")

scored_xgb = cf_test.select(["custid", "message"]).with_columns(p_buy)

best_prob_xgb = (
    scored_xgb.sort(["custid", "p_buy"], descending=[False, True])
    .group_by("custid")
    .head(1)
    .select(["custid", "message", "p_buy"])
    .rename({"message": "best_msg_prob_xgb"})
)

best_prob_xgb.head(10)

custid,best_msg_prob_xgb,p_buy
str,str,f32
"""U649014""","""endurance""",0.003373
"""U3183951""","""endurance""",0.005569
"""U3177511""","""strength""",0.02025
"""U1119364""","""strength""",0.001705
"""U1493262""","""endurance""",0.007288
"""U1081210""","""strength""",0.042678
"""U2134133""","""endurance""",0.001924
"""U2882222""","""endurance""",0.002146
"""U1720522""","""strength""",0.099894


We trained an XGBoost model in pyrsm to predict P(buyer=yes | X, message) and tuned it using 3-fold cross-validation over multiple hyperparameter settings (including at least two hyperparameters: max_depth and n_estimators, plus learning_rate). We selected the combination with the highest mean CV AUC (0.8864; best params: max_depth=3, n_estimators=600, learning_rate=0.05). For each customer, we scored all seven actions (six messages plus control) and chose the message with the highest predicted purchase probability.

---
***Q2(XGBOOST)***

In [ ]:
# XGB Q2: Share of customers for whom each message maximizes predicted purchase probability
q2_share_xgb = (
    best_prob_xgb.group_by("best_msg_prob_xgb")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum()).alias("share"))
    .sort("share", descending=True)
)

q2_share_xgb

best_msg_prob_xgb,n,share
str,u32,f64
"""endurance""",125157,0.695317
"""strength""",52951,0.294172
"""water""",1407,0.007817
"""racquet""",375,0.002083
"""team""",70,0.000389
"""control""",28,0.000156
"""backcountry""",12,0.000067


Under the tuned XGBoost model, the recommended message distribution is concentrated in two messages: endurance (69.53%) and strength (29.42%). The remaining messages are rarely optimal—water (0.78%), racquet (0.21%), team (0.04%), control/no-message (0.02%), and backcountry (≈0.00%). This indicates that, according to XGBoost, most customers’ predicted purchase probability is maximized by either endurance or strength, with limited incremental value from the other messages.

---
***Q3(XGBOOST)***

In [ ]:
# XGB Q3: Best message by expected profit (uses XGB p_buy + same spend OLS model)
margin = 0.4

actions = (
    pentathlon_nptb.select(pl.col("message").unique())
    .to_series()
    .to_list()
)

freq_cols = [c for c in pentathlon_nptb.columns if c.startswith("freq_")]
messages_no_control = [m for m in actions if m != "control"]

def add_int_cols(df: pl.DataFrame) -> pl.DataFrame:
    exprs = []
    for m in messages_no_control:
        msg_ind = (pl.col("message") == m).cast(pl.Float64)
        for f in freq_cols:
            exprs.append((msg_ind * pl.col(f)).alias(f"int_{m}_{f}"))
    return df.with_columns(exprs)

# Spend model (buyers only): OLS on log1p(total_os)
buyers_train = train.filter(pl.col("buyer") == "yes").with_columns(
    pl.col("total_os").cast(pl.Float64).log1p().alias("log_total_os")
)
buyers_train = add_int_cols(buyers_train)

int_cols = [c for c in buyers_train.columns if c.startswith("int_")]
spend_cols = ["message", "age", "female", "income", "education", "children"] + freq_cols + int_cols

buyers_pd = buyers_train.select(spend_cols + ["log_total_os"]).to_pandas()
formula = "log_total_os ~ C(message) + C(age) + C(female) + income + education + children + " + " + ".join(freq_cols + int_cols)
spend_ols = smf.ols(formula, data=buyers_pd).fit()

# Counterfactual rows on test: customers × messages
base_test = test.select(["custid", "age", "female", "income", "education", "children"] + freq_cols)
cf_test = base_test.join(pl.DataFrame({"message": actions}), how="cross")

# XGB p_buy (requires xgb_best and predict_proba from Q1)
p_buy = predict_proba(xgb_best, cf_test).rename("p_buy")

# Spend prediction m_buy
cf_test_int = add_int_cols(cf_test)
cf_pd_spend = cf_test_int.select(spend_cols).to_pandas()
log_m_buy = spend_ols.predict(cf_pd_spend)
m_buy = np.expm1(np.clip(log_m_buy, a_min=0, a_max=None))

# Expected profit and best message
scored_ep_xgb = cf_test.select(["custid", "message"]).with_columns(
    p_buy,
    pl.Series("m_buy", m_buy),
).with_columns(
    (pl.col("p_buy") * margin * pl.col("m_buy")).alias("ep")
)

best_ep_xgb = (
    scored_ep_xgb.sort(["custid", "ep"], descending=[False, True])
    .group_by("custid")
    .head(1)
    .select(["custid", "message", "ep"])
    .rename({"message": "best_msg_profit_xgb"})
)

best_ep_xgb.head(10)

custid,best_msg_profit_xgb,ep
str,str,f64
"""U1709501""","""endurance""",0.037198
"""U3206274""","""endurance""",9.475685
"""U747546""","""water""",0.209134
"""U3316471""","""endurance""",0.018336
"""U2184388""","""water""",0.019318
"""U499260""","""endurance""",0.018021
"""U1599545""","""endurance""",0.01831
"""U3370410""","""racquet""",0.033044
"""U3292034""","""endurance""",0.019962


We compute expected profit by combining (1) XGBoost’s predicted purchase probability P(buyer=yes | X, message) with (2) a conditional order-size model estimated on buyers only. We fit an OLS regression for log1p(total_os) using customer features and message indicators (including message-by-frequency interaction terms), back-transform predictions to Euros, and calculate expected profit as EP = P(buy) × 0.4 × E(total_os | buy) (COGS = 60% ⇒ margin = 40%). For each customer, we score all messages (including control/no-message) and select the message that maximizes expected profit.

---
***Q4(XGBOOST)***

In [ ]:
# XGB  Q4: Share of customers for whom each message maximizes expected profit
q4_share_xgb = (
    best_ep_xgb.group_by("best_msg_profit_xgb")
    .agg(pl.len().alias("n"))
    .with_columns((pl.col("n") / pl.col("n").sum()).alias("share"))
    .sort("share", descending=True)
)

q4_share_xgb

best_msg_profit_xgb,n,share
str,u32,f64
"""endurance""",113855,0.632528
"""water""",43921,0.244006
"""strength""",12230,0.067944
"""backcountry""",6614,0.036744
"""racquet""",3249,0.01805
"""team""",131,0.000728


Under the XGBoost profit-maximizing policy, recommendations are concentrated in endurance (63.25%) and water (24.40%), with smaller segments assigned to strength (6.79%), backcountry (3.67%), and racquet (1.81%). Team is rarely optimal (0.07%), and control/no-message is effectively never selected under expected profit maximization. Overall, the distribution indicates meaningful heterogeneity in profit response, with most value driven by endurance and water.

---
***Q5(XGBOOST)***

In [ ]:
# XGB Q5: Average expected profit per customer under personalization
q5_avg_ep_xgb = best_ep_xgb.select(pl.col("ep").mean().alias("avg_ep_per_customer"))
q5_avg_ep_xgb

avg_ep_per_customer
f64
0.48303


Under the personalized XGBoost policy, we assign each customer the message (or control) that maximizes their expected profit and then average this maximum expected profit across customers. The resulting expected profit is 0.4830 Euros per customer on the test set.

---
***Q6(XGBOOST)***

In [ ]:
# XGB Q6: Expected profit per customer if everyone receives the same message (or control)
q6_ep_by_message_xgb = (
    scored_ep_xgb.group_by("message")
    .agg(pl.col("ep").mean().alias("avg_ep_per_customer"))
    .sort("avg_ep_per_customer", descending=True)
)

q6_ep_by_message_xgb

message,avg_ep_per_customer
str,f64
"""endurance""",0.45484
"""water""",0.445795
"""backcountry""",0.421698
"""strength""",0.419798
"""team""",0.394367
"""racquet""",0.385275
"""control""",0.308906


If every customer must receive the same message, the best single-message choice is endurance, with an expected profit of 0.4548 Euros per customer. The next best are water (0.4458) and backcountry (0.4217), while control/no-message is lowest at 0.3089. Expected profit varies across messages, and the best one-size-fits-all policy still underperforms the personalized XGBoost policy (Q5).

---
***Q7(XGBOOST)***

In [ ]:
# XGB Q7: Expected profit per customer under random assignment to one of 7 actions (incl. control)
q7_random_ep_xgb = (
    scored_ep_xgb.group_by("message")
    .agg(pl.col("ep").mean().alias("avg_ep"))
    .select(pl.col("avg_ep").mean().alias("avg_ep_per_customer_random"))
)

q7_random_ep_xgb

avg_ep_per_customer_random
f64
0.404383


Under random assignment (each customer equally likely to receive any of the 6 messages or the control option), the expected profit is 0.4044 Euros per customer. This is lower than the best single-message policy (Q6) and lower than the personalized XGBoost policy (Q5), since random assignment frequently assigns customers to lower-performing messages and the no-message condition.

---
***Q8(XGBOOST)***

In [ ]:
# XGB Q8: Scale to 5,000,000 customers and compute improvements vs (a) best single, (b) random, (c) control
N = 5_000_000

ep_personal = 0.48303
ep_best_one = 0.45484    # best single message from Q6 (endurance)
ep_random   = 0.404383
ep_control  = 0.308906

summary_xgb = pl.DataFrame({
    "scenario": ["personalized", "best_single_message", "random_assignment", "control_no_message"],
    "ep_per_customer": [ep_personal, ep_best_one, ep_random, ep_control],
}).with_columns(
    (pl.col("ep_per_customer") * N).alias("total_ep_5m")
)

improvement_xgb = pl.DataFrame({
    "baseline": ["best_single_message", "random_assignment", "control_no_message"],
    "baseline_ep_per_customer": [ep_best_one, ep_random, ep_control],
}).with_columns(
    ((ep_personal - pl.col("baseline_ep_per_customer")) * N).alias("euro_gain_5m"),
    ((ep_personal - pl.col("baseline_ep_per_customer")) / pl.col("baseline_ep_per_customer")).alias("pct_gain")
)

summary_xgb, improvement_xgb

(shape: (4, 3)
 ┌─────────────────────┬─────────────────┬─────────────┐
 │ scenario            ┆ ep_per_customer ┆ total_ep_5m │
 │ ---                 ┆ ---             ┆ ---         │
 │ str                 ┆ f64             ┆ f64         │
 ╞═════════════════════╪═════════════════╪═════════════╡
 │ personalized        ┆ 0.48303         ┆ 2.41515e6   │
 │ best_single_message ┆ 0.45484         ┆ 2.2742e6    │
 │ random_assignment   ┆ 0.404383        ┆ 2.021915e6  │
 │ control_no_message  ┆ 0.308906        ┆ 1.54453e6   │
 └─────────────────────┴─────────────────┴─────────────┘,
 shape: (3, 4)
 ┌─────────────────────┬──────────────────────────┬──────────────┬──────────┐
 │ baseline            ┆ baseline_ep_per_customer ┆ euro_gain_5m ┆ pct_gain │
 │ ---                 ┆ ---                      ┆ ---          ┆ ---      │
 │ str                 ┆ f64                      ┆ f64          ┆ f64      │
 ╞═════════════════════╪══════════════════════════╪══════════════╪══════════╡
 │ best_s

Using the test-set expected profit estimates, we scale to a typical blast of 5,000,000 customers by multiplying expected profit per customer by 5,000,000. The resulting total expected profit is:

Personalized: 0.48303 × 5,000,000 = €2,415,150

Best single message (endurance): 0.45484 × 5,000,000 = €2,274,200

Random assignment (status quo): 0.404383 × 5,000,000 = €2,021,915

Control (no message): 0.308906 × 5,000,000 = €1,544,530

Improvements from personalization:

vs best single message: +€140,950 (+6.20%)

vs random assignment: +€393,235 (+19.45%)

vs control/no message: +€870,620 (+56.37%)

Overall, the personalized XGBoost policy delivers the highest expected profit compared to the status quo (random assignment) and the no-message baseline, and it also improves over the best one-size-fits-all message.

ALL ML: Total Expected Profit for 5,000,000 Customers (Euros)

| Model | Personalized Total (€) | Best Single Total (€) | Random Total (€) | Control Total (€) |
|---|---:|---:|---:|---:|
| Logistic Regression | 2,440,515 | 2,299,870 | 2,044,325 | 1,561,005 |
| Neural Network | 2,584,695 | 2,413,055 | 2,132,900 | 1,581,075 |
| Random Forest | 2,483,080 | 2,220,625 | 2,017,630 | 1,643,630 |
| XGBoost (pyrsm) | 2,415,150 | 2,274,200 | 2,021,915 | 1,544,530 |

Across 5,000,000 customers, the Neural Network delivers the highest total expected profit under personalization (€2,584,695), followed by Random Forest (€2,483,080), Logistic Regression (€2,440,515), and XGBoost (€2,415,150). For all four models, the ranking of policies is consistent: personalized > best single message > random assignment > no-message control, indicating that personalized targeting reliably generates higher total profit than one-size-fits-all or status-quo approaches.

---
**A NEW Policy Proposal**

The proposal is a good step toward using expected-profit modeling, but it has a few weaknesses. First, forcing the top two messages split 50/50 is not profit-optimal: if the top message is much better than the second, the rule leaves money on the table. Second, it ignores the no-message option even when predicted profit is low or negative, which can lead to unnecessary contact and lower long-term customer value. Third, re-running the model using only the first three weeks of the month may introduce selection bias (customers already contacted may behave differently), and the policy does not specify any monitoring/guardrails for model drift or fairness across departments.

Improvement: replace the fixed 50/50 split with an expected-profit weighted rule (e.g., allocate shares proportional to each message’s predicted expected profit, and allow 100% to the top message when the gap is large). Also include a no-message threshold (send nothing when expected profit is below 0 or below a small positive cutoff), and track weekly performance with holdout/control monitoring to detect drift before updating the policy. 